---

# Section 0: Configuration Globale et Imports

---


## 0.1 Imports des Bibliothèques


In [13]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import json
import os
import time
from collections import Counter
# Imports scipy pour matrices sparse
from scipy.sparse import csr_matrix, coo_matrix
# Imports LightFM
try:
    from lightfm import LightFM
    from lightfm.data import Dataset
    from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
    from lightfm.cross_validation import random_train_test_split
    print("✅ LightFM installé et disponible")
    LIGHTFM_AVAILABLE = True
except ImportError:
    print("❌ LightFM n'est pas installé!")
    print("   Installer avec: pip install lightfm")
    LIGHTFM_AVAILABLE = False
    raise ImportError("LightFM est requis pour ce notebook")
# Configuration des warnings et affichage
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
# Style des visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print("\n✅ Imports terminés")


✅ LightFM installé et disponible

✅ Imports terminés


## 0.2 Configuration Globale du Projet


In [14]:
# ============================================================================
# PARAMÈTRES GLOBAUX - À MODIFIER SELON VOS BESOINS
# ============================================================================

# Taille du sample (nombre de transactions à échantillonner)
SAMPLE_SIZE = '50K'  # Options: 1000, 10000, 50000

# Stratégie de sampling
MIN_USER_TRANSACTIONS = 5   # Users avec au moins N transactions (dans dataset complet)
MIN_ITEM_TRANSACTIONS = 10  # Items avec au moins N transactions (dans dataset complet)

# Features sélectionnées
ITEM_FEATURE_COLUMNS = [
    'product_group_name',
    'product_type_name',
    'garment_group_name',
    'colour_group_name'
]
USER_FEATURE_COLUMNS = [
    'age_group',
    'club_member_status',
    'fashion_news_frequency'
]

# Paramètres de split
TEMPORAL_TRAIN_RATIO = 0.8  # 80% train, 20% test pour split temporel
RANDOM_TEST_PERCENTAGE = 0.2  # 20% test pour split aléatoire
USERBASED_TRAIN_RATIO = 0.8  # 80% users train, 20% users test

# Choix de la stratégie de split pour l'entraînement final
SPLIT_STRATEGY = 'temporal'  # Options: 'temporal', 'random', 'userbased'

# Paramètres d'entraînement
N_EPOCHS = 10
N_THREADS = 4

# Paramètres de grid search
GRID_SEARCH_PARAMS = {
    'no_components': [10, 30, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1],
    'item_alpha': [1e-6, 1e-5, 1e-4],
    'user_alpha': [1e-6, 1e-5, 1e-4]
}

# Métriques d'évaluation
K_VALUES = [5, 10, 20]  # Pour Precision@K, Recall@K

# Seed pour reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Chemins des données
DATA_PATH = 'data/'
TRANSACTIONS_FILE = DATA_PATH + 'transactions_train.csv'
ARTICLES_FILE = DATA_PATH + 'articles.csv'
CUSTOMERS_FILE = DATA_PATH + 'customers.csv'

# ============================================================================
# AFFICHAGE DE LA CONFIGURATION
# ============================================================================
print("="*80)
print("CONFIGURATION DU PROJET")
print("="*80)
print(f"\n📊 Dataset:")
print(f"   • Taille du sample: {SAMPLE_SIZE} transactions")
print(f"   • Min transactions/user (filtrage): {MIN_USER_TRANSACTIONS}")
print(f"   • Min transactions/item (filtrage): {MIN_ITEM_TRANSACTIONS}")
print(f"\n🎯 Features:")
print(f"   • Item features: {len(ITEM_FEATURE_COLUMNS)} colonnes")
print(f"   • User features: {len(USER_FEATURE_COLUMNS)} colonnes")
print(f"\n🔀 Split Strategy:")
print(f"   • Stratégie choisie: {SPLIT_STRATEGY.upper()}")
print(f"   • Temporal ratio: {TEMPORAL_TRAIN_RATIO*100:.0f}% train")
print(f"   • Random test: {RANDOM_TEST_PERCENTAGE*100:.0f}%")
print(f"\n🤖 Entraînement:")
print(f"   • Epochs: {N_EPOCHS}")
print(f"   • Threads: {N_THREADS}")
print(f"   • Random state: {RANDOM_STATE}")
print(f"\n🔍 Grid Search:")
print(f"   • Paramètres à tester:")
for param, values in GRID_SEARCH_PARAMS.items():
    print(f"     - {param}: {values}")
total_combinations = np.prod([len(v) for v in GRID_SEARCH_PARAMS.values()])
print(f"   • Total combinaisons: {total_combinations}")
print(f"\n✅ Configuration chargée")
print("="*80)


CONFIGURATION DU PROJET

📊 Dataset:
   • Taille du sample: 50K transactions
   • Min transactions/user (filtrage): 5
   • Min transactions/item (filtrage): 10

🎯 Features:
   • Item features: 4 colonnes
   • User features: 3 colonnes

🔀 Split Strategy:
   • Stratégie choisie: TEMPORAL
   • Temporal ratio: 80% train
   • Random test: 20%

🤖 Entraînement:
   • Epochs: 10
   • Threads: 4
   • Random state: 42

🔍 Grid Search:
   • Paramètres à tester:
     - no_components: [10, 30, 50, 100]
     - learning_rate: [0.01, 0.05, 0.1]
     - item_alpha: [1e-06, 1e-05, 0.0001]
     - user_alpha: [1e-06, 1e-05, 0.0001]
   • Total combinaisons: 108

✅ Configuration chargée


## 0.3 Fonctions Utilitaires


In [15]:
def print_section_header(title, section_number=None):
    """Affiche un header de section formaté."""
    print("\n" + "="*80)
    if section_number:
        print(f"SECTION {section_number}: {title.upper()}")
    else:
        print(title.upper())
    print("="*80 + "\n")

def print_subsection_header(title):
    """Affiche un header de sous-section formaté."""
    print("\n" + "-"*80)
    print(title)
    print("-"*80 + "\n")

def print_dataframe_info(df, name):
    """Affiche des informations sur un DataFrame."""
    print(f"\n📊 {name}:")
    print(f"   • Shape: {df.shape}")
    print(f"   • Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Colonnes: {list(df.columns)}")

def calculate_sparsity(n_interactions, n_users, n_items):
    """Calcule la sparsité d'une matrice user-item."""
    return 1 - (n_interactions / (n_users * n_items))

def format_large_number(num):
    """Formate un grand nombre avec des séparateurs."""
    return f"{num:,}"

def timer(func):
    """Décorateur pour mesurer le temps d'exécution."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"\n⏱️  Temps d'exécution: {end-start:.2f} secondes")
        return result
    return wrapper

def evaluate_model(model, test_interactions, train_interactions=None, 
                   item_features=None, user_features=None, k=10):
    """Évalue un modèle LightFM avec plusieurs métriques."""
    metrics = {}
    # Precision@K
    precision = precision_at_k(
        model, test_interactions, 
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features,
        k=k
    ).mean()
    metrics[f'precision@{k}'] = precision
    # Recall@K
    recall = recall_at_k(
        model, test_interactions,
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features,
        k=k
    ).mean()
    metrics[f'recall@{k}'] = recall
    # AUC
    auc = auc_score(
        model, test_interactions,
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features
    ).mean()
    metrics['auc'] = auc
    return metrics
print("✅ Fonctions utilitaires chargées")


✅ Fonctions utilitaires chargées


---

# Section 3: Prétraitement et Construction LightFM

---


In [16]:
# ⚙️ CONFIGURATION : Choisir la taille du sample


print(f"{'='*80}")
print(f"CONFIGURATION")
print(f"{'='*80}")
print(f"\n✅ Taille sélectionnée : {SAMPLE_SIZE}")
print(f"   Chemin source : data/sampled/{SAMPLE_SIZE}/")
print(f"   Chemin destination : data/processed/{SAMPLE_SIZE}/")
# Vérifier que la taille existe
import os
sample_path = f'data/sampled/{SAMPLE_SIZE}/'
if not os.path.exists(sample_path):
    print(f"\n❌ ERREUR: Le dossier {sample_path} n'existe pas!")
    print(f"   Exécutez d'abord Step 2 - Section 9 pour créer les samples.")
    raise FileNotFoundError(f"Sample {SAMPLE_SIZE} non trouvé")
print(f"\n✅ Configuration validée")


CONFIGURATION

✅ Taille sélectionnée : 50K
   Chemin source : data/sampled/50K/
   Chemin destination : data/processed/50K/

✅ Configuration validée


## 3.1 Chargement des Données Échantillonnées


In [17]:
# Chemins basés sur SAMPLE_SIZE
SAMPLED_DATA_PATH = f'data/sampled/{SAMPLE_SIZE}/'
PROCESSED_DATA_PATH = f'data/processed/{SAMPLE_SIZE}/'

# Créer dossier processed si nécessaire
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
print(f"📂 Chargement des données échantillonnées (Step 2 - {SAMPLE_SIZE})...")
print("-" * 80)

# Charger transactions
transactions = pd.read_csv(SAMPLED_DATA_PATH + 'transactions_sampled.csv')
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
print(f"✓ Transactions: {len(transactions):,} lignes")

# Charger articles
articles = pd.read_csv(SAMPLED_DATA_PATH + 'articles_sampled.csv')
print(f"✓ Articles: {len(articles):,} lignes")

# Charger customers
customers = pd.read_csv(SAMPLED_DATA_PATH + 'customers_sampled.csv')
print(f"✓ Customers: {len(customers):,} lignes")

# Charger métadonnées de sampling
with open(SAMPLED_DATA_PATH + 'sampling_metadata.json', 'r') as f:
    sampling_metadata = json.load(f)

print(f"✓ Métadonnées de sampling chargées")
print("\n" + "=" * 80)
print(f"📊 STATISTIQUES DU DATASET ÉCHANTILLONNÉ ({SAMPLE_SIZE})")
print("=" * 80)
print(f"Transactions: {len(transactions):,}")
print(f"Utilisateurs uniques: {transactions['customer_id'].nunique():,}")
print(f"Articles uniques: {transactions['article_id'].nunique():,}")
print(f"Période: {transactions['t_dat'].min().date()} → {transactions['t_dat'].max().date()}")
print(f"Durée: {(transactions['t_dat'].max() - transactions['t_dat'].min()).days} jours")
print(f"\n📋 Info de sampling:")
print(f"   Stratégie: {sampling_metadata.get('strategy', 'N/A')}")
print(f"   Créé le: {sampling_metadata.get('creation_date', 'N/A')}")
print(f"   Avg trans/user: {sampling_metadata['statistics']['avg_trans_per_user']:.2f}")
print(f"   Avg trans/item: {sampling_metadata['statistics']['avg_trans_per_item']:.2f}")
print("\n✅ Données chargées avec succès")


📂 Chargement des données échantillonnées (Step 2 - 50K)...
--------------------------------------------------------------------------------
✓ Transactions: 50,000 lignes
✓ Articles: 24,216 lignes
✓ Customers: 46,668 lignes
✓ Métadonnées de sampling chargées

📊 STATISTIQUES DU DATASET ÉCHANTILLONNÉ (50K)
Transactions: 50,000
Utilisateurs uniques: 46,668
Articles uniques: 24,216
Période: 2018-09-20 → 2020-09-22
Durée: 733 jours

📋 Info de sampling:
   Stratégie: Stratégie Combinée (users≥5 + items≥10)
   Créé le: 2025-10-29 09:18:40
   Avg trans/user: 1.07
   Avg trans/item: 2.06

✅ Données chargées avec succès


## 3.2 Construction du Dataset LightFM avec ID Mappings

### 📋 Approche Recommandée par LightFM

Selon la documentation officielle ([Building datasets](https://making.lyst.com/lightfm/docs/examples/dataset.html)), LightFM utilise la classe **`Dataset`** pour :

1. **Créer automatiquement les mappings** user_id/item_id → indices consécutifs (0, 1, 2, ...)
2. **Gérer les features** (item features et user features)
3. **Construire les matrices d'interactions** au format scipy.sparse

### 🔑 Pourquoi cette approche ?

**Problème sans Dataset** :
- `customer_id` : Hash hexadécimal de 64 caractères (ex: `000058a12d5b43e67d...`)
- `article_id` : Entiers non consécutifs (ex: 663713001, 108775015)
- Nécessité de mappings manuels → risque d'erreurs, incompatibilité avec LightFM

**Solution avec Dataset.fit()** :
- LightFM gère automatiquement les mappings internes
- Garantit la cohérence entre interactions et features
- Format optimisé pour l'entraînement

### 📖 Référence Documentation

```python
from lightfm.data import Dataset
dataset = Dataset()
dataset.fit(
    users=(user_id for user_id in all_user_ids),
    items=(item_id for item_id in all_item_ids)
)
```

Source : LightFM Documentation - "Building the ID mappings"


In [18]:
print("=" * 80)
print("CRÉATION DU DATASET LIGHTFM ET MAPPINGS")
print("=" * 80)

# Créer l'objet Dataset
dataset = Dataset()

# Récupérer les IDs uniques
unique_users = transactions['customer_id'].unique()
unique_items = transactions['article_id'].unique()
print(f"\n📊 Nombre d'IDs uniques à mapper:")
print(f"   Utilisateurs: {len(unique_users):,}")
print(f"   Articles: {len(unique_items):,}")

# Préparer les features pour le fit
print(f"\n🔄 Préparation des features pour fit()...")
# Item features : récupérer toutes les valeurs uniques
item_feature_columns = [
    'product_group_name',
    'index_group_name',
    'garment_group_name',
    'colour_group_name'
]

# Filtrer articles pour le sample
articles_filtered = articles[articles['article_id'].isin(unique_items)].copy()

# Imputer valeurs manquantes AVANT de récupérer les features
for col in item_feature_columns:
    articles_filtered[col].fillna('Unknown', inplace=True)

# Récupérer toutes les features items uniques
all_item_features = set()
for col in item_feature_columns:
    unique_values = articles_filtered[col].unique()
    for val in unique_values:
        all_item_features.add(f"{col}:{val}")
print(f"   ✓ Item features uniques: {len(all_item_features):,}")
print(f"     Exemple: {list(all_item_features)[:5]}")

# User features : préparer les features users
customers_filtered = customers[customers['customer_id'].isin(unique_users)].copy()

# Imputer age et créer age_group
median_age = customers_filtered['age'].median()
customers_filtered['age'].fillna(median_age, inplace=True)
customers_filtered['age_group'] = pd.cut(
    customers_filtered['age'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['<25', '25-35', '35-45', '45-55', '55+'],
    include_lowest=True
).astype(str)

# Imputer autres features
customers_filtered['club_member_status'].fillna('ACTIVE', inplace=True)
customers_filtered['fashion_news_frequency'].fillna('NONE', inplace=True)

# Récupérer toutes les features users uniques
user_feature_columns = [
    'age_group',
    'club_member_status',
    'fashion_news_frequency'
]
all_user_features = set()
for col in user_feature_columns:
    unique_values = customers_filtered[col].unique()
    for val in unique_values:
        all_user_features.add(f"{col}:{val}")
print(f"   ✓ User features uniques: {len(all_user_features):,}")
print(f"     Exemple: {list(all_user_features)[:5]}")

# FIT du Dataset avec users, items, et features
print(f"\n🔄 Fitting du Dataset LightFM...")
print(f"   Cette étape crée les mappings internes...")
dataset.fit(
    users=unique_users,
    items=unique_items,
    item_features=all_item_features,
    user_features=all_user_features
)
print(f"   ✓ Dataset fitted avec succès!")

# Vérifier les dimensions
num_users, num_items = dataset.interactions_shape()
print(f"\n📊 Dimensions du Dataset:")
print(f"   Users: {num_users:,}")
print(f"   Items: {num_items:,}")
print(f"   Item features: {len(all_item_features):,}")
print(f"   User features: {len(all_user_features):,}")

# Créer des mappings inverses pour référence (optionnel, pour notre usage)
# Ces mappings nous permettent de récupérer les IDs originaux
user_id_mapping, user_features_mapping, item_id_mapping, item_features_mapping = dataset.mapping()
print(f"\n✅ Mappings créés avec succès")
print(f"   user_id_mapping: {len(user_id_mapping):,} entrées")
print(f"   item_id_mapping: {len(item_id_mapping):,} entrées")


CRÉATION DU DATASET LIGHTFM ET MAPPINGS

📊 Nombre d'IDs uniques à mapper:
   Utilisateurs: 46,668
   Articles: 24,216

🔄 Préparation des features pour fit()...
   ✓ Item features uniques: 91
     Exemple: ['product_group_name:Items', 'colour_group_name:Light Red', 'colour_group_name:Off White', 'product_group_name:Furniture', 'colour_group_name:Other Red']
   ✓ User features uniques: 11
     Exemple: ['age_group:35-45', 'age_group:<25', 'club_member_status:ACTIVE', 'age_group:55+', 'fashion_news_frequency:NONE']

🔄 Fitting du Dataset LightFM...
   Cette étape crée les mappings internes...
   ✓ Dataset fitted avec succès!

📊 Dimensions du Dataset:
   Users: 46,668
   Items: 24,216
   Item features: 91
   User features: 11

✅ Mappings créés avec succès
   user_id_mapping: 46,668 entrées
   item_id_mapping: 24,216 entrées


## 3.3 Construction de la Matrice d'Interactions User-Item

### 📊 Approche LightFM : `build_interactions()`

Selon la documentation officielle, LightFM utilise `dataset.build_interactions()` pour créer la matrice sparse :

```python
(interactions, weights) = dataset.build_interactions(
    ((user_id, item_id) for user_id, item_id in interaction_pairs)
)
```

**Avantages** :
- Utilise automatiquement les mappings créés par `fit()`
- Retourne une matrice sparse COO optimisée
- Gère automatiquement les poids (optionnels)
- Format directement compatible avec `LightFM.fit()`

### 📖 Référence Documentation

Source : LightFM Documentation - "Building the interactions matrix"


In [19]:
print("=" * 80)
print("CONSTRUCTION DE LA MATRICE D'INTERACTIONS AVEC LIGHTFM")
print("=" * 80)

# Préparer les interactions au format (user_id, item_id)
print(f"\n🔄 Construction de la matrice complète...")
print(f"   Nombre d'interactions: {len(transactions):,}")

# Construire la matrice avec build_interactions()
(user_item_matrix, weights_matrix) = dataset.build_interactions(
    ((row['customer_id'], row['article_id']) 
     for idx, row in transactions.iterrows())
)
print(f"   ✓ Matrice construite")
print(f"\n📊 Caractéristiques de la matrice:")
print(f"   Type: {type(user_item_matrix)}")
print(f"   Format: {user_item_matrix.format}")
print(f"   Shape: {user_item_matrix.shape}")
print(f"   Non-zéros (nnz): {user_item_matrix.nnz:,}")
print(f"   Dtype: {user_item_matrix.dtype}")

# Calculer sparsité
n_users, n_items = user_item_matrix.shape
sparsity = 1 - (user_item_matrix.nnz / (n_users * n_items))
print(f"\n⚠️  Sparsité: {sparsity:.4%}")
print(f"   Densité: {(1-sparsity):.4%}")

# Convertir en CSR pour opérations efficaces
user_item_matrix_csr = user_item_matrix.tocsr()
print(f"\n💾 Mémoire:")
print(f"   Matrice sparse (CSR): {user_item_matrix_csr.data.nbytes / 1024**2:.2f} MB")

# Comparer avec dense (hypothétique)
dense_size = (n_users * n_items * 4) / 1024**3  # 4 bytes pour float32
if dense_size < 1000:  # Seulement si < 1TB
    print(f"   Matrice dense (hypothétique): {dense_size:.2f} GB")
    print(f"   Économie de mémoire: {(1 - (user_item_matrix_csr.data.nbytes / (n_users * n_items * 4))) * 100:.2f}%")
else:
    print(f"   Matrice dense: > 1 TB (impossible à charger en mémoire)")

# Statistiques par utilisateur
interactions_per_user = np.array(user_item_matrix_csr.sum(axis=1)).flatten()
print(f"\n📈 Statistiques par utilisateur:")
print(f"   Moyenne: {interactions_per_user.mean():.2f} interactions")
print(f"   Médiane: {np.median(interactions_per_user):.0f} interactions")
print(f"   Min/Max: {interactions_per_user.min():.0f} / {interactions_per_user.max():.0f}")

# Statistiques par article
interactions_per_item = np.array(user_item_matrix_csr.sum(axis=0)).flatten()
print(f"\n📈 Statistiques par article:")
print(f"   Moyenne: {interactions_per_item.mean():.2f} interactions")
print(f"   Médiane: {np.median(interactions_per_item):.0f} interactions")
print(f"   Min/Max: {interactions_per_item.min():.0f} / {interactions_per_item.max():.0f}")
print("\n✅ Matrice user-item construite avec LightFM")


CONSTRUCTION DE LA MATRICE D'INTERACTIONS AVEC LIGHTFM

🔄 Construction de la matrice complète...
   Nombre d'interactions: 50,000
   ✓ Matrice construite

📊 Caractéristiques de la matrice:
   Type: <class 'scipy.sparse._coo.coo_matrix'>
   Format: coo
   Shape: (46668, 24216)
   Non-zéros (nnz): 50,000
   Dtype: int32

⚠️  Sparsité: 99.9956%
   Densité: 0.0044%

💾 Mémoire:
   Matrice sparse (CSR): 0.19 MB
   Matrice dense (hypothétique): 4.21 GB
   Économie de mémoire: 100.00%

📈 Statistiques par utilisateur:
   Moyenne: 1.07 interactions
   Médiane: 1 interactions
   Min/Max: 1 / 4

📈 Statistiques par article:
   Moyenne: 2.06 interactions
   Médiane: 1 interactions
   Min/Max: 1 / 72

✅ Matrice user-item construite avec LightFM


## 3.4 Construction des Matrices de Features avec LightFM

### 📋 Approche LightFM : `build_item_features()` et `build_user_features()`

Selon la documentation officielle, LightFM utilise des fonctions dédiées pour construire les matrices de features :

```python
# Item features : (item_id, [list_of_features])
item_features = dataset.build_item_features(
    ((item_id, [feature1, feature2, ...]) for item_id in items)
)

# User features : (user_id, [list_of_features])
user_features = dataset.build_user_features(
    ((user_id, [feature1, feature2, ...]) for user_id in users)
)
```

**Format des features** :
- Chaque feature est une **string** au format `"column:value"`
- Exemple : `"product_group_name:Garment Upper body"`, `"age_group:25-35"`
- LightFM gère automatiquement l'encodage one-hot en interne

### 📖 Référence Documentation

Source : LightFM Documentation - "Building the features matrices"


In [20]:
print("=" * 80)
print("CONSTRUCTION DES MATRICES DE FEATURES AVEC LIGHTFM")
print("=" * 80)

# ============================================================================
# 5.1 ITEM FEATURES
# ============================================================================
print(f"\n{'='*80}")
print("5.1 ITEM FEATURES")
print(f"{'='*80}")
print(f"\n📋 Features items sélectionnées:")
for col in item_feature_columns:
    n_unique = articles_filtered[col].nunique()
    print(f"   • {col:25s}: {n_unique:3d} catégories")

# Préparer les features au format LightFM : (item_id, [list_of_features])
print(f"\n🔄 Préparation des features au format LightFM...")
item_features_list = []
for idx, row in articles_filtered.iterrows():
    article_id = row['article_id']
    features = []
    for col in item_feature_columns:
        feature_value = row[col]
        features.append(f"{col}:{feature_value}")
    item_features_list.append((article_id, features))
print(f"   ✓ {len(item_features_list):,} items préparés")
print(f"\n   Exemple (3 premiers items):")
for item_id, features in item_features_list[:3]:
    print(f"   • {item_id}: {features[:2]}...")

# Construire la matrice avec LightFM
print(f"\n🔄 Construction de la matrice item_features...")
item_features_matrix = dataset.build_item_features(item_features_list)
print(f"   ✓ Matrice construite")
print(f"\n📊 Caractéristiques de la matrice item features:")
print(f"   Type: {type(item_features_matrix)}")
print(f"   Format: {item_features_matrix.format}")
print(f"   Shape: {item_features_matrix.shape}")
print(f"   Non-zéros (nnz): {item_features_matrix.nnz:,}")
print(f"   Mémoire: {item_features_matrix.data.nbytes / 1024**2:.2f} MB")

# ============================================================================
# 5.2 USER FEATURES
# ============================================================================
print(f"\n{'='*80}")
print("5.2 USER FEATURES")
print(f"{'='*80}")
print(f"\n📋 Features users sélectionnées:")
for col in user_feature_columns:
    n_unique = customers_filtered[col].nunique()
    print(f"   • {col:25s}: {n_unique:2d} catégories")
print(f"\n📊 Distribution des features users:")
print(f"\n   Age groups:")
age_dist = customers_filtered['age_group'].value_counts().sort_index()
for group, count in age_dist.items():
    pct = count / len(customers_filtered) * 100
    print(f"   • {group:8s}: {count:6,} ({pct:5.1f}%)")

# Préparer les features au format LightFM
print(f"\n🔄 Préparation des features au format LightFM...")
user_features_list = []
for idx, row in customers_filtered.iterrows():
    customer_id = row['customer_id']
    features = []
    for col in user_feature_columns:
        feature_value = row[col]
        features.append(f"{col}:{feature_value}")
    user_features_list.append((customer_id, features))
print(f"   ✓ {len(user_features_list):,} users préparés")
print(f"\n   Exemple (3 premiers users):")
for user_id, features in user_features_list[:3]:
    print(f"   • {user_id[:20]}...: {features}")

# Construire la matrice avec LightFM
print(f"\n🔄 Construction de la matrice user_features...")
user_features_matrix = dataset.build_user_features(user_features_list)
print(f"   ✓ Matrice construite")
print(f"\n📊 Caractéristiques de la matrice user features:")
print(f"   Type: {type(user_features_matrix)}")
print(f"   Format: {user_features_matrix.format}")
print(f"   Shape: {user_features_matrix.shape}")
print(f"   Non-zéros (nnz): {user_features_matrix.nnz:,}")
print(f"   Mémoire: {user_features_matrix.data.nbytes / 1024**2:.2f} MB")
print("\n✅ Matrices de features construites avec LightFM")


CONSTRUCTION DES MATRICES DE FEATURES AVEC LIGHTFM

5.1 ITEM FEATURES

📋 Features items sélectionnées:
   • product_group_name       :  15 catégories
   • index_group_name         :   5 catégories
   • garment_group_name       :  21 catégories
   • colour_group_name        :  50 catégories

🔄 Préparation des features au format LightFM...
   ✓ 24,216 items préparés

   Exemple (3 premiers items):
   • 108775015: ['product_group_name:Garment Upper body', 'index_group_name:Ladieswear']...
   • 108775044: ['product_group_name:Garment Upper body', 'index_group_name:Ladieswear']...
   • 110065001: ['product_group_name:Underwear', 'index_group_name:Ladieswear']...

🔄 Construction de la matrice item_features...
   ✓ Matrice construite

📊 Caractéristiques de la matrice item features:
   Type: <class 'scipy.sparse._csr.csr_matrix'>
   Format: csr
   Shape: (24216, 24307)
   Non-zéros (nnz): 121,080
   Mémoire: 0.46 MB

5.2 USER FEATURES

📋 Features users sélectionnées:
   • age_group            

---

# Section 8: Modèle Hybride et Analyse

---


## 8.4 Entraînement du Modèle Hybride

### 🎯 Configuration

Nous utilisons les **mêmes hyperparamètres** que le modèle CF pur (Section 6) pour une comparaison équitable.

La seule différence : ajout de `item_features` lors de l'entraînement.


In [21]:
print("=" * 80)
print("CRÉATION DES MATRICES TRAIN/TEST AVEC L'API LIGHTFM")
print("=" * 80)

# ============================================================================
# APPROCHE CORRECTE : Utiliser l'API LightFM pour créer train/test
# ============================================================================

print(f"\n🔄 Split temporel des données...")

# 1. Trier les transactions par date
transactions_sorted = transactions.sort_values('t_dat')

# 2. Split 80/20
split_idx = int(len(transactions_sorted) * 0.8)
train_data = transactions_sorted.iloc[:split_idx].copy()
test_data = transactions_sorted.iloc[split_idx:].copy()

print(f"   ✓ Train: {len(train_data):,} transactions")
print(f"   ✓ Test: {len(test_data):,} transactions")

# 3. Binariser (garder une seule occurrence par paire user-item)
train_data = train_data.drop_duplicates(subset=['customer_id', 'article_id'])
test_data = test_data.drop_duplicates(subset=['customer_id', 'article_id'])

print(f"\n🔄 Après binarisation:")
print(f"   ✓ Train: {len(train_data):,} interactions uniques")
print(f"   ✓ Test: {len(test_data):,} interactions uniques")

# 4. Filtrer test pour ne garder que les interactions nouvelles
train_pairs = set(zip(train_data['customer_id'], train_data['article_id']))
test_pairs = list(zip(test_data['customer_id'], test_data['article_id']))
mask = [pair not in train_pairs for pair in test_pairs]
test_data = test_data[mask]

print(f"\n🔄 Après filtrage des doublons train/test:")
print(f"   ✓ Test: {len(test_data):,} interactions nouvelles")

# 5. Construire les matrices avec l'API LightFM (utilise le dataset déjà créé)
print(f"\n🔄 Construction des matrices avec LightFM...")

(train_interactions, _) = dataset.build_interactions(
    ((row['customer_id'], row['article_id']) for _, row in train_data.iterrows())
)

(test_interactions, _) = dataset.build_interactions(
    ((row['customer_id'], row['article_id']) for _, row in test_data.iterrows())
)

print(f"   ✓ train_interactions: {train_interactions.shape} - {train_interactions.nnz:,} nnz")
print(f"   ✓ test_interactions: {test_interactions.shape} - {test_interactions.nnz:,} nnz")

# 6. Convertir en CSR pour compatibilité
train_interactions = train_interactions.tocsr()
test_interactions = test_interactions.tocsr()

print(f"\n✅ Matrices créées avec l'API LightFM")
print(f"\n🔍 Vérification d'alignement:")
print(f"   train_interactions.shape[0] (users): {train_interactions.shape[0]:,}")
print(f"   user_features_matrix.shape[0] (users): {user_features_matrix.shape[0]:,}")
print(f"   train_interactions.shape[1] (items): {train_interactions.shape[1]:,}")
print(f"   item_features_matrix.shape[0] (items): {item_features_matrix.shape[0]:,}")

assert train_interactions.shape[0] == user_features_matrix.shape[0], "Désalignement users !"
assert train_interactions.shape[1] == item_features_matrix.shape[0], "Désalignement items !"

print(f"✅ Alignement parfait garanti !")

CRÉATION DES MATRICES TRAIN/TEST AVEC L'API LIGHTFM

🔄 Split temporel des données...
   ✓ Train: 40,000 transactions
   ✓ Test: 10,000 transactions

🔄 Après binarisation:
   ✓ Train: 39,996 interactions uniques
   ✓ Test: 10,000 interactions uniques

🔄 Après filtrage des doublons train/test:
   ✓ Test: 10,000 interactions nouvelles

🔄 Construction des matrices avec LightFM...
   ✓ train_interactions: (46668, 24216) - 39,996 nnz
   ✓ test_interactions: (46668, 24216) - 10,000 nnz

✅ Matrices créées avec l'API LightFM

🔍 Vérification d'alignement:
   train_interactions.shape[0] (users): 46,668
   user_features_matrix.shape[0] (users): 46,668
   train_interactions.shape[1] (items): 24,216
   item_features_matrix.shape[0] (items): 24,216
✅ Alignement parfait garanti !


In [22]:
print("=" * 80)
print("ENTRAÎNEMENT MODÈLE HYBRIDE")
print("=" * 80)

# Créer le modèle hybride
hybrid_model = LightFM(
    loss='warp',
    no_components=55,
    learning_rate=0.00394802143257261,
    item_alpha=1.93e-08,
    user_alpha=1.89e-08,
    random_state=42
)

print(f"\n🔄 Entraînement en cours...")
print(f"   Avec item_features ET user_features (LightFM)")

import time
start_time = time.time()

# Entraînement avec les features LightFM
hybrid_model.fit(
    interactions=train_interactions,      # ← Matrices LightFM !
    item_features=item_features_matrix,   # ← Features LightFM (alignées) !
    user_features=user_features_matrix,   # ← Features LightFM (alignées) !
    epochs=10,
    num_threads=4,
    verbose=True
)

training_time = time.time() - start_time
print(f"\n✅ Entraînement terminé en {training_time:.1f}s")

ENTRAÎNEMENT MODÈLE HYBRIDE

🔄 Entraînement en cours...
   Avec item_features ET user_features (LightFM)
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9

✅ Entraînement terminé en 1.2s


## 8.5 Comparaison CF Pur vs Hybrid Model

### 🎯 Objectif

Évaluer si l'ajout de features améliore les performances.

### 📊 Métriques

- Precision@K, Recall@K, AUC sur le test set
- Coverage (diversité du catalogue)


In [23]:
print("=" * 80)
print("ÉVALUATION DU MODÈLE HYBRIDE")
print("=" * 80)

K_VALUES = [5, 10, 20]

print(f"\n🔄 Évaluation du modèle hybride (K={K_VALUES})...")
results_comparison = {'hybrid': {}}

print(f"\n2️⃣  HYBRID MODEL (avec item + user features LightFM):")
print(f"{'K':<6} | {'Precision@K':<13} | {'Recall@K':<13} | {'AUC':<10}")
print("-" * 55)

for k in K_VALUES:
    # ✅ CORRECTION : Passer AUSSI user_features lors de l'évaluation
    prec = precision_at_k(
        hybrid_model,
        test_interactions,                   # ← Matrice LightFM
        k=k,
        train_interactions=train_interactions,  # ← Matrice LightFM
        item_features=item_features_matrix,     # ← Features LightFM
        user_features=user_features_matrix,     # ✅ AJOUTÉ !
        num_threads=4
    ).mean()

    rec = recall_at_k(
        hybrid_model,
        test_interactions,
        k=k,
        train_interactions=train_interactions,
        item_features=item_features_matrix,
        user_features=user_features_matrix,     # ✅ AJOUTÉ !
        num_threads=4
    ).mean()

    auc = auc_score(
        hybrid_model,
        test_interactions,
        train_interactions=train_interactions,
        item_features=item_features_matrix,
        user_features=user_features_matrix,     # ✅ AJOUTÉ !
        num_threads=4
    ).mean()

    results_comparison['hybrid'][k] = {
        'precision': prec,
        'recall': rec,
        'auc': auc
    }
    print(f"{k:<6} | {prec:<13.4f} | {rec:<13.4f} | {auc:<10.4f}")

print(f"\n✅ Évaluation terminée")

ÉVALUATION DU MODÈLE HYBRIDE

🔄 Évaluation du modèle hybride (K=[5, 10, 20])...

2️⃣  HYBRID MODEL (avec item + user features LightFM):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0006        | 0.0032        | 0.5577    
10     | 0.0004        | 0.0039        | 0.5577    
20     | 0.0004        | 0.0070        | 0.5577    

✅ Évaluation terminée


## 8.6 Analyse Cold-Start

### 🎯 Objectif

Vérifier si le modèle hybride performe mieux sur les **items avec peu d'interactions** (cold-start).

### 📊 Segmentation Items

- **Populaires** : >P75 interactions train
- **Moyens** : P25-P75 interactions
- **Cold-start** : <P25 interactions


In [24]:
print("=" * 80)
print("ANALYSE COLD-START")
print("=" * 80)
# Convertir les matrices en CSR pour permettre l'indexation
train_interactions_csr = train_interactions.tocsr()
test_interactions_csr = test_interactions.tocsr()
# Utiliser test_interactions de Step 4
# Calculer la popularité des items (nombre d'interactions train)
item_popularity = np.array(train_interactions.sum(axis=0)).flatten()
# Statistiques
print(f"\n📊 Distribution des interactions par item (train):")
print(f"   Min : {item_popularity.min()}")
print(f"   Q25 : {np.percentile(item_popularity, 25):.0f}")
print(f"   Q50 : {np.percentile(item_popularity, 50):.0f}")
print(f"   Q75 : {np.percentile(item_popularity, 75):.0f}")
print(f"   Max : {item_popularity.max()}")
# Définir les seuils
q25 = np.percentile(item_popularity, 25)
q75 = np.percentile(item_popularity, 75)
# Créer les segments d'items
item_segments = {
    'cold_start': np.where(item_popularity < q25)[0],
    'moyens': np.where((item_popularity >= q25) & (item_popularity < q75))[0],
    'populaires': np.where(item_popularity >= q75)[0]
}
print(f"\n📦 Segments d'items créés:")
for seg_name, items in item_segments.items():
    print(f"   • {seg_name.capitalize():<15} : {len(items):>6,} items ({len(items)/num_items*100:>5.1f}%)")
# Pour chaque segment, calculer les métriques
print(f"\n🔄 Évaluation par segment d'items...")
# On doit filtrer les interactions test par segment d'item
K_COLDSTART = 10
print(f"\n{'Segment':<15} | {'N Items':<10} | {'CF Pure P@10':<15} | {'Hybrid P@10':<15} | {'Amélioration':<15}")
print("-" * 90)
coldstart_results = {}
for seg_name, item_indices in item_segments.items():
    if len(item_indices) == 0:
        continue
    # Filtrer test_interactions pour ne garder que les items du segment (utiliser CSR)
    test_segment = test_interactions_csr[:, item_indices]
    test_segment_hybrid = test_interactions_csr[:, item_indices]
    # Vérifier qu'il y a des interactions
    if test_segment.nnz == 0:
        print(f"{seg_name.capitalize():<15} | {len(item_indices):>10,} | {'N/A':<15} | {'N/A':<15} | {'N/A':<15}")
        continue
    # Évaluer CF pur
    try:
        cf_prec = precision_at_k(cf_pure_model, test_segment, k=K_COLDSTART,
                                train_interactions=train_interactions, num_threads=4).mean()
    except:
        cf_prec = 0.0
    # Évaluer Hybrid
    try:
        hybrid_prec = precision_at_k(hybrid_model, test_segment_hybrid, k=K_COLDSTART,
                                    train_interactions=train_interactions,
                                    item_features=item_features_matrix, num_threads=4).mean()
    except:
        hybrid_prec = 0.0
    improvement = hybrid_prec - cf_prec
    coldstart_results[seg_name] = {
        'n_items': len(item_indices),
        'cf_pure_prec': cf_prec,
        'hybrid_prec': hybrid_prec,
        'improvement': improvement
    }
    print(f"{seg_name.capitalize():<15} | {len(item_indices):>10,} | {cf_prec:<15.4f} | {hybrid_prec:<15.4f} | {improvement:>+14.4f}")
print(f"\n✅ Analyse cold-start terminée")
print(f"\n💡 INTERPRÉTATION:")
print(f"   Si Hybrid > CF Pure sur segment 'cold_start', cela indique que")
print(f"   les features aident à généraliser aux items avec peu d'historique.")


ANALYSE COLD-START

📊 Distribution des interactions par item (train):
   Min : 0
   Q25 : 1
   Q50 : 1
   Q75 : 2
   Max : 64

📦 Segments d'items créés:
   • Cold_start      :  4,131 items ( 17.1%)
   • Moyens          : 11,831 items ( 48.9%)
   • Populaires      :  8,254 items ( 34.1%)

🔄 Évaluation par segment d'items...

Segment         | N Items    | CF Pure P@10    | Hybrid P@10     | Amélioration   
------------------------------------------------------------------------------------------
Cold_start      |      4,131 | 0.0000          | 0.0000          |        +0.0000
Moyens          |     11,831 | 0.0000          | 0.0000          |        +0.0000
Populaires      |      8,254 | 0.0000          | 0.0000          |        +0.0000

✅ Analyse cold-start terminée

💡 INTERPRÉTATION:
   Si Hybrid > CF Pure sur segment 'cold_start', cela indique que
   les features aident à généraliser aux items avec peu d'historique.
